# Books Web Scraping using Python

## 1. Import Libraries

In [1]:
# import the important libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd

## 2. Fetch Website Content

In [2]:
# fetch HTML content using url
url = 'https://books.toscrape.com/'
response = requests.get(url)
response.status_code

200

## 3. Parse HTML using BeautifulSoup

In [3]:
# parse HTML using BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')
print(soup.title.text)


    All products | Books to Scrape - Sandbox



## 4. Crawl Books Across Multiple Pages

In [4]:
# Define a function to recursively crawl all available pages

data_list = []
def crawl_next_page(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    for i in soup.select('ol.row li'):
        data_dict = {}

        # extract name, price, rating and stock_status
        data_dict['name'] = i.select('h3')[-1].text.strip()
        data_dict['price'] = i.select('.product_price p.price_color')[0].text.strip().replace('Â£', '')
        data_dict['rating'] = i.select('p.star-rating')[0].get('class')[-1]
        data_dict['stock_status'] = i.select('p.instock.availability')[0].text.strip()
        data_list.append(data_dict)
        
    # next page button
    if soup.select_one('li.next a'):
        if 'catalogue' in soup.select_one('li.next a').get('href'):
            next_url = 'https://books.toscrape.com/' + soup.select_one('li.next a').get('href')
        else:
            next_url = 'https://books.toscrape.com/catalogue/' + soup.select_one('li.next a').get('href')

        # calling recursive function
        crawl_next_page(next_url)

    return data_list

In [5]:
# Let's start crawling
crawl_next_page('https://books.toscrape.com/')

[{'name': 'A Light in the ...',
  'price': '51.77',
  'rating': 'Three',
  'stock_status': 'In stock'},
 {'name': 'Tipping the Velvet',
  'price': '53.74',
  'rating': 'One',
  'stock_status': 'In stock'},
 {'name': 'Soumission',
  'price': '50.10',
  'rating': 'One',
  'stock_status': 'In stock'},
 {'name': 'Sharp Objects',
  'price': '47.82',
  'rating': 'Four',
  'stock_status': 'In stock'},
 {'name': 'Sapiens: A Brief History ...',
  'price': '54.23',
  'rating': 'Five',
  'stock_status': 'In stock'},
 {'name': 'The Requiem Red',
  'price': '22.65',
  'rating': 'One',
  'stock_status': 'In stock'},
 {'name': 'The Dirty Little Secrets ...',
  'price': '33.34',
  'rating': 'Four',
  'stock_status': 'In stock'},
 {'name': 'The Coming Woman: A ...',
  'price': '17.93',
  'rating': 'Three',
  'stock_status': 'In stock'},
 {'name': 'The Boys in the ...',
  'price': '22.60',
  'rating': 'Four',
  'stock_status': 'In stock'},
 {'name': 'The Black Maria',
  'price': '52.15',
  'rating': 'On

In [6]:
# Length of the data_list
print(f'The length of the data_list is {len(data_list)}.')

The length of the data_list is 1000.


## 5. Convert Data to DataFrame

In [7]:
# Let's convert the data_list to the Pandas DataFrame
books_df = pd.DataFrame(data_list)
books_df.head()

,name,price,rating,stock_status
0,A Light in the ...,51.77,Three,In stock
1,Tipping the Velvet,53.74,One,In stock
2,Soumission,50.10,One,In stock
3,Sharp Objects,47.82,Four,In stock
4,Sapiens: A Brief History ...,54.23,Five,In stock


## 6. Data Understanding

In [8]:
# Shape of the data
print(f'There are {books_df.shape[0]} rows and {books_df.shape[1]} columns in the books_df.')

There are 1000 rows and 4 columns in the books_df.


In [9]:
# Data types of the books_df
books_df.dtypes

name            str
price           str
rating          str
stock_status    str
dtype: object

In [10]:
# convert the data type of 'Price' from str to float
books_df['price'] = books_df['price'].astype('float')
books_df.dtypes

name                str
price           float64
rating              str
stock_status        str
dtype: object

In [11]:
# Let's map the ratings

books_df["rating"] = books_df["rating"].map({'One':1,
                                             'Two':2,
                                             'Three':3,
                                             'Four':4,
                                             'Five': 5})
books_df.head()

,name,price,rating,stock_status
0,A Light in the ...,51.77,3,In stock
1,Tipping the Velvet,53.74,1,In stock
2,Soumission,50.10,1,In stock
3,Sharp Objects,47.82,4,In stock
4,Sapiens: A Brief History ...,54.23,5,In stock


In [12]:
# Info of the data
books_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   name          1000 non-null   str    
 1   price         1000 non-null   float64
 2   rating        1000 non-null   int64  
 3   stock_status  1000 non-null   str    
dtypes: float64(1), int64(1), str(2)
memory usage: 31.4 KB


In [13]:
# Statistical summary of the data
books_df.describe()

,price,rating
count,1000.00000,1000.000000
mean,35.07035,2.923000
std,14.44669,1.434967
min,10.00000,1.000000
25%,22.10750,2.000000
50%,35.98000,3.000000
75%,47.45750,4.000000
max,59.99000,5.000000


## 7. Export Data to CSV

In [14]:
# export this data to csv
books_df.to_csv("books_to_scrape.csv", index=False)

## 8. Basic Data Analysis

In [15]:
# no. of unique books
print(f'There are {books_df['name'].nunique()} unique books.')

There are 992 unique books.


In [16]:
# Top 10 Expensive books
books_df.sort_values(by = 'price', ascending = False).head(10)

,name,price,rating,stock_status
648,The Perfect Play (Play ...,59.99,3,In stock
617,Last One Home (New ...,59.98,3,In stock
860,Civilization and Its Discontents,59.95,2,In stock
560,The Barefoot Contessa Cookbook,59.92,5,In stock
366,The Diary of a ...,59.90,3,In stock
657,The Bone Hunters (Lexy ...,59.71,3,In stock
133,Thomas Jefferson and the ...,59.64,1,In stock
387,Boar Island (Anna Pigeon ...,59.48,3,In stock
549,The Man Who Mistook ...,59.45,4,In stock
393,The Improbability of Love,59.45,1,In stock


In [17]:
# Top 10 Cheapest books
books_df.sort_values(by = 'price', ascending = True).head(10)

,name,price,rating,stock_status
638,An Abundance of Katherines,10.00,5,In stock
501,The Origin of Species,10.01,4,In stock
716,The Tipping Point: How ...,10.02,2,In stock
84,Patience,10.16,3,In stock
302,Greek Mythic History,10.23,5,In stock
558,The Fellowship of the ...,10.27,2,In stock
479,History of Beauty,10.29,4,In stock
242,The Lucifer Effect: Understanding ...,10.40,1,In stock
434,NaNo What Now? Finding ...,10.41,4,In stock
274,Pet Sematary,10.56,3,In stock


In [18]:
# Rating Analysis
print('Average Price by Rating')
books_df.groupby('rating')['price'].mean()

Average Price by Rating


rating
1    34.561195
2    34.810918
3    34.692020
4    36.093296
5    35.374490
Name: price, dtype: float64

In [19]:
print('No. of books by rating')
books_df.rating.value_counts().sort_values(ascending = False)

No. of books by rating


rating
1    226
3    203
5    196
2    196
4    179
Name: count, dtype: int64

In [20]:
# stock status
books_df['stock_status'].value_counts()

stock_status
In stock    1000
Name: count, dtype: int64

**All 1000 books are in stock.**

## Conclusion

In this project, I used Python and BeautifulSoup to scrape books data across multiple pages and converted the extracted information into a structured dataset for analysis.

Key concepts covered:
- HTML parsing
- Pagination handling
- Data extraction
- Export data using pandas
- Basic data analysis